## Import

In [1]:
import random
import requests
from datetime import datetime
import time
import os
from dotenv import load_dotenv
import pandas as pd
import logging
from pathlib import Path
from tqdm import tqdm
from sqlalchemy import create_engine

load_dotenv()
log_path = Path("/app/logs/error.log")
log_path.parent.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.ERROR,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_path, encoding="utf-8"),
        logging.StreamHandler()
    ],
    force=True,
)

TABLE = 'model_comparison_per_stock'
PG_URI = os.getenv('POSTGRES_URI')

In [2]:
df_pred_stock = pd.read_sql(f"SELECT * FROM {TABLE}", create_engine(PG_URI))
df_pred_stock.head()

,model,train_r2,train_rmse,test_r2,test_rmse,pred_mean_ret,actual_mean_ret,next_day_pred_ret,stock_code,run_date,category,train_start,train_end,test_start,test_end
0,Naive,-0.977590,0.033227,-1.200769,0.030311,0.001685,0.003159,0.025260,0050,2026-06-10,ETF,2020-01-01,2026-05-09,2026-05-10,2026-06-10
1,MA(5),-0.202439,0.025943,-0.236072,0.022716,0.004878,0.003159,-0.003962,0050,2026-06-10,ETF,2020-01-01,2026-05-09,2026-05-10,2026-06-10
2,AR(5),0.000462,0.023653,-0.014802,0.020582,0.000551,0.003159,0.000909,0050,2026-06-10,ETF,2020-01-01,2026-05-09,2026-05-10,2026-06-10
3,Naive,-0.973351,0.037878,-0.821330,0.026565,0.001202,0.002561,0.022090,0052,2026-06-10,ETF,2020-01-01,2026-05-09,2026-05-10,2026-06-10
4,MA(5),-0.198612,0.029555,-0.260476,0.022099,0.005053,0.002561,-0.008022,0052,2026-06-10,ETF,2020-01-01,2026-05-09,2026-05-10,2026-06-10


## MAE 計算與視覺化

對四種模型（Naive、MA(5)、AR(5)、LSTM）分別計算 test MAE。  
由於 DB 只存 RMSE，這裡對 Baseline 三種模型從原始股票資料重新跑預測取得逐日誤差；LSTM 直接用 DB 的 `test_rmse` 近似標示（因無逐日預測值）。

In [ ]:
import numpy as np
import sys
sys.path.insert(0, '/app')
from model.baseline import _split
from sqlalchemy import create_engine

TEST_START = '2026-05-10'
TRAIN_START = '2020-01-01'
TRAIN_END = '2026-06-12'


def _fetch_ret_series(stock_code: str) -> pd.Series:
    """從 DB 撈指定股票的 close 報酬率序列（DatetimeIndex）。"""
    engine = create_engine(PG_URI)
    query = f"""
        SELECT date, close
        FROM "daily_info_{stock_code}"
        WHERE stock_code_id = '{stock_code}'
          AND date BETWEEN '{TRAIN_START}' AND '{TRAIN_END}'
        ORDER BY date ASC
    """
    df = pd.read_sql(query, engine)
    if df.empty:
        return pd.Series(dtype=float)
    df['date'] = pd.to_datetime(df['date'])
    ret = df.set_index('date')['close'].astype(float).pct_change().dropna()
    return ret


def compute_mae_baseline(df_pred_stock: pd.DataFrame) -> pd.DataFrame:
    """
    從原始股價資料重新計算 Naive / MA(5) / AR(5) 的 test MAE。
    預測目標與 feature_engineering.py 一致：close 的 pct_change（ret_wide）。
    """
    from statsmodels.tsa.ar_model import AutoReg

    stock_codes = df_pred_stock['stock_code'].unique().tolist()
    rows = []
    errors = []

    for code in stock_codes:
        try:
            ret = _fetch_ret_series(code)
        except Exception as e:
            errors.append(f'{code}: fetch failed — {e}')
            continue
        if len(ret) < 30:
            errors.append(f'{code}: too short ({len(ret)} rows)')
            continue

        train, test = _split(ret, TEST_START)
        if len(test) == 0:
            errors.append(f'{code}: test set empty after split at {TEST_START}')
            continue

        # Naive: y_pred[t] = ret[t-1]
        shifted = ret.shift(1).reindex(test.index).dropna()
        y_true_n = test.reindex(shifted.index)
        rows.append({'model': 'Naive', 'stock_code': code,
                     'mae': float(np.abs(y_true_n.values - shifted.values).mean())})

        # MA(5): y_pred[t] = mean(ret[t-5:t-1])
        rolled = ret.rolling(5).mean().shift(1).reindex(test.index).dropna()
        y_true_m = test.reindex(rolled.index)
        rows.append({'model': 'MA(5)', 'stock_code': code,
                     'mae': float(np.abs(y_true_m.values - rolled.values).mean())})

        # AR(5)
        try:
            model_ar = AutoReg(pd.Series(train.values, dtype=float), lags=5).fit()
            start, end = len(train), len(train) + len(test) - 1
            y_pred_ar = pd.Series(model_ar.predict(start=start, end=end).values, index=test.index)
            y_true_ar = test.iloc[:len(y_pred_ar)]
            rows.append({'model': 'AR(5)', 'stock_code': code,
                         'mae': float(np.abs(y_true_ar.values - y_pred_ar.values).mean())})
        except Exception as e:
            errors.append(f'{code} AR(5): {e}')

    if errors:
        print(f"略過 {len(errors)} 筆，前 5 筆錯誤：")
        for msg in errors[:5]:
            print(' ', msg)

    if not rows:
        raise RuntimeError("所有股票都失敗，rows 為空。請確認 DB 連線與欄位名稱。")

    return pd.DataFrame(rows).groupby('model')['mae'].mean().reset_index()


def compute_mae_all_models(df_pred_stock: pd.DataFrame) -> pd.DataFrame:
    """
    回傳四種 model 的 MAE DataFrame。
    LSTM 因 DB 無逐日預測值，以 test_rmse 均值作為上界估計標示。
    """
    df_baseline_mae = compute_mae_baseline(df_pred_stock)

    lstm_rmse_mean = df_pred_stock[df_pred_stock['model'] == 'LSTM']['test_rmse'].mean()
    df_lstm = pd.DataFrame([{
        'model': 'LSTM',
        'mae': lstm_rmse_mean,
        'is_estimate': True,
    }])

    return pd.concat([df_baseline_mae.assign(is_estimate=False), df_lstm],
                     ignore_index=True)


print("計算 MAE 中，請稍候...")
df_mae = compute_mae_all_models(df_pred_stock)
df_mae

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

MODEL_ORDER = ['Naive', 'MA(5)', 'AR(5)', 'LSTM']
palette = {'Naive': '#4C72B0', 'MA(5)': '#55A868', 'AR(5)': '#C44E52', 'LSTM': '#DD8452'}

df_plot = df_mae.set_index('model').reindex(MODEL_ORDER).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    df_plot['model'],
    df_plot['mae'],
    color=[palette[m] for m in df_plot['model']],
    width=0.5,
    edgecolor='white',
    linewidth=1.2,
)

# 數值標籤
for bar, (_, row) in zip(bars, df_plot.iterrows()):
    label = f"{row['mae']:.5f}"
    if row['is_estimate']:
        label += '\n(RMSE est.)'
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + df_plot['mae'].max() * 0.01,
        label,
        ha='center', va='bottom', fontsize=9,
    )

# 虛線框標示 LSTM 是估計值
lstm_idx = df_plot[df_plot['model'] == 'LSTM'].index[0]
bars[lstm_idx].set_linestyle('--')
bars[lstm_idx].set_edgecolor('#DD8452')
bars[lstm_idx].set_linewidth(2)

ax.set_title('Test MAE by Model (averaged across all stocks)', fontsize=13, pad=12)
ax.set_xlabel('Model')
ax.set_ylabel('MAE (return)')
ax.set_ylim(0, df_plot['mae'].max() * 1.25)
ax.spines[['top', 'right']].set_visible(False)

note = mpatches.Patch(facecolor='none', edgecolor='gray', linestyle='--',
                      label='LSTM: RMSE used as upper-bound estimate (no per-day predictions in DB)')
ax.legend(handles=[note], fontsize=8, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
df_pred_stock[df_pred_stock['stock_code']=='2330']

,model,train_r2,train_rmse,test_r2,test_rmse,pred_mean_ret,actual_mean_ret,next_day_pred_ret,stock_code,run_date,category,train_start,train_end,test_start,test_end
123,Naive,-1.031246,0.027222,-1.174285,0.024084,-0.000161,0.00043,0.004357,2330,2026-06-10,半導體業,2020-01-01,2026-05-09,2026-05-10,2026-06-10
124,MA(5),-0.199855,0.020938,-0.313838,0.018722,0.002638,0.00043,-0.006243,2330,2026-06-10,半導體業,2020-01-01,2026-05-09,2026-05-10,2026-06-10
125,AR(5),0.002965,0.019086,-0.004947,0.016374,0.001516,0.00043,0.003382,2330,2026-06-10,半導體業,2020-01-01,2026-05-09,2026-05-10,2026-06-10
160,LSTM,0.640562,0.011462,-1.865990,0.027485,0.011403,0.00043,0.010228,2330,2026-06-10,半導體業,2020-01-01,2026-05-09,2026-05-10,2026-06-10


## END